# X-SDD — 01: Verifikasi dataset & pembuatan split
Langkah: (1) temukan folder dataset, (2) scan & validasi gambar, (3) cek jumlah per kelas vs paper,
(4) cek duplikat, (5) lihat contoh gambar per kelas, (6) buat split stratified 70/30 dan simpan ke CSV.

In [ ]:
import os, json, hashlib, random
from pathlib import Path
from collections import Counter

import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42          # seed KHUSUS untuk split (tetap untuk semua run & semua varian)
TEST_SIZE = 0.30
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
EXPECTED_COUNTS = [63, 122, 134, 203, 203, 238, 397]   # dari paper X-SDD, diurutkan (tanpa nama kelas)
OUT_DIR = Path("/kaggle/working")

# Isi manual jika auto-detect gagal, contoh: "/kaggle/input/xsdd/X-SDD"
DATA_ROOT = None

In [ ]:
def find_data_root_candidates(base="/kaggle/input"):
    """Cari folder yang punya >= 2 sub-folder berisi gambar (kandidat folder 'per kelas')."""
    cands = []
    for dirpath, dirnames, _ in os.walk(base):
        class_dirs = []
        for d in dirnames:
            sub = Path(dirpath, d)
            if any(Path(f).suffix.lower() in IMG_EXT for f in os.listdir(sub)):
                class_dirs.append(d)
        if len(class_dirs) >= 2:
            cands.append((dirpath, sorted(class_dirs)))
    return cands

if DATA_ROOT is None:
    cands = find_data_root_candidates()
    for path, cls in cands:
        print(path, "->", cls)
    assert len(cands) == 1, "Kandidat tidak tunggal. Isi DATA_ROOT secara manual."
    DATA_ROOT = cands[0][0]
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
root = Path(DATA_ROOT)
class_names = sorted(d.name for d in root.iterdir() if d.is_dir())      # urutan alfabet = urutan ImageFolder
class_to_idx = {c: i for i, c in enumerate(class_names)}

rows, bad = [], []
for c in class_names:
    for p in sorted((root / c).rglob("*")):
        if p.suffix.lower() not in IMG_EXT:
            continue
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                w, h, mode = im.size[0], im.size[1], im.mode
        except Exception as e:
            bad.append((str(p), repr(e)))
            continue
        rows.append(dict(
            rel_path=str(p.relative_to(root)), label=c, label_idx=class_to_idx[c],
            width=w, height=h, mode=mode, md5=hashlib.md5(p.read_bytes()).hexdigest(),
        ))
df = pd.DataFrame(rows)

print("class_to_idx:", class_to_idx)
print("Total gambar terbaca:", len(df), "| file rusak:", len(bad))
for b in bad[:10]:
    print("  RUSAK:", b)

In [ ]:
# --- Cek jumlah per kelas vs paper (dibandingkan sebagai himpunan angka, bukan nama kelas) ---
counts = df.groupby(["label_idx", "label"]).size().rename("n").reset_index()
print(counts.to_string(index=False))
print("\nUkuran gambar :", Counter(zip(df.width, df.height)))
print("Mode warna    :", Counter(df["mode"]))

got = sorted(counts["n"].tolist())
print("\nJumlah per kelas (urut):", got)
print("Harapan paper (urut)   :", EXPECTED_COUNTS)
print("Total:", len(df), "(harapan 1360)")
print("COCOK DENGAN PAPER" if got == EXPECTED_COUNTS else ">>> TIDAK COCOK, periksa dulu sebelum lanjut <<<")

In [ ]:
# --- Cek duplikat (potensi kebocoran train/test) ---
dup = df[df.duplicated("md5", keep=False)]
print("Gambar yang duplikat persis (md5):", len(dup))
if len(dup):
    cross = dup.groupby("md5")["label"].nunique()
    print("Grup duplikat lintas kelas:", int((cross > 1).sum()))
    print(dup.sort_values("md5")[["rel_path", "label", "md5"]].head(20).to_string(index=False))

In [ ]:
# --- Contoh gambar per kelas: untuk memastikan indeks -> nama kelas (bandingkan dengan Fig. 3 paper Symmetry) ---
import matplotlib.pyplot as plt
rng = random.Random(0)
fig, axes = plt.subplots(len(class_names), 5, figsize=(10, 2.0 * len(class_names)))
for i, c in enumerate(class_names):
    paths = df[df.label == c].rel_path.tolist()
    for j, rp in enumerate(rng.sample(paths, min(5, len(paths)))):
        ax = axes[i, j]
        ax.imshow(Image.open(root / rp).convert("RGB"))
        ax.axis("off")
        if j == 0:
            ax.set_title(f"{i}: {c} (n={len(paths)})", fontsize=8, loc="left")
plt.tight_layout()
plt.show()

In [ ]:
# --- Split stratified 70/30 dengan seed tetap ---
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["label_idx"], random_state=SPLIT_SEED, shuffle=True
)
df["split"] = "train"
df.loc[test_df.index, "split"] = "test"

summary = (df.pivot_table(index=["label_idx", "label"], columns="split", values="rel_path", aggfunc="count")
             .fillna(0).astype(int))
summary["total"] = summary["train"] + summary["test"]
summary["test_%"] = (100 * summary["test"] / summary["total"]).round(1)
print(summary.to_string())
print("\nTrain:", (df.split == "train").sum(), "| Test:", (df.split == "test").sum(), "(harapan 952 / 408)")

# Cek kebocoran duplikat lintas split
leak = set(df[df.split == "train"].md5) & set(df[df.split == "test"].md5)
print("md5 yang muncul di train DAN test:", len(leak))

assert set(df[df.split == "train"].rel_path).isdisjoint(set(df[df.split == "test"].rel_path))

In [ ]:
# --- Simpan split (path relatif -> tidak bergantung lokasi mount Kaggle) ---
split_csv = OUT_DIR / f"xsdd_split_seed{SPLIT_SEED}.csv"
df[["rel_path", "label", "label_idx", "split"]].to_csv(split_csv, index=False)
with open(OUT_DIR / "xsdd_class_to_idx.json", "w") as f:
    json.dump(class_to_idx, f, indent=2)

print("Tersimpan:", split_csv)
print("md5 file split:", hashlib.md5(split_csv.read_bytes()).hexdigest())